# פרק 7: בניית אפליקציות צ'אט
## התחלה מהירה עם Microsoft Foundry Models API

פנקס זה מותאם מתוך [מאגר הדוגמאות של Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) הכולל פנקסים שמבצעים גישה לשירותי [Azure OpenAI](aoai-assignment.ipynb).

> **הערה:** GitHub Models ייסגר בסוף יולי 2026. פנקס זה פונה כעת ל-[Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), המציע את אותו קטלוג מודלים חינמי לנסות ואת חוויית Azure AI Inference SDK.


# סקירה  
"מודלי שפה גדולים הם פונקציות שממפות טקסט לטקסט. בהתחשב במחרוזת קלט של טקסט, מודל שפה גדול מנסה לנבא את הטקסט שיבוא לאחר מכן" (1). המחברת "התחלה מהירה" הזו תציג למשתמשים מושגים ברמת על של LLM, דרישות חבילה מרכזיות לצורך התחלה עם AML, מבוא רך לעיצוב פרומפט, וכמה דוגמאות קצרות של מקרים שונים של שימוש. 


## תוכן העניינים  

[סקירה כללית](#overview)  
[כיצד להשתמש בשירות OpenAI](#how-to-use-openai-service)  
[1. יצירת שירות OpenAI שלך](#1.-creating-your-openai-service)  
[2. התקנה](#2.-installation)    
[3. אישורים](#3.-credentials)  

[מקרי שימוש](#use-cases)    
[1. סיכום טקסט](#1.-summarize-text)  
[2. סיווג טקסט](#2.-classify-text)  
[3. יצירת שמות חדשים למוצרים](#3.-generate-new-product-names)  
[4. כוונון עדין של מסווג](#4.fine-tune-a-classifier)  

[הפניות](#references)


### בנו את הפקודה הראשונה שלכם  
תרגיל קצר זה יספק מבוא בסיסי לשליחת פקודות למודל ב-Microsoft Foundry Models עבור משימה פשוטה של "סיכום".  


**שלבים**:  
1. התקינו את ספריית `azure-ai-inference` בסביבת הפייתון שלכם, אם עדיין לא עשיתם זאת.  
2. טענו את ספריות העזר הסטנדרטיות והגדירו את פרטי ההתחברות שלכם ל-Microsoft Foundry Models.  
3. בחרו מודל למשימה שלכם  
4. צרו פקודה פשוטה עבור המודל  
5. שלחו את הבקשה שלכם אל ה-API של המודל!  


### 1. התקנת `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. ייבוא ספריות עזר ויצירת אישורי גישה


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. מציאת הדגם המתאים  
דגמים כמו GPT-4o ו-GPT-4o mini יכולים להבין וליצור שפה טבעית, וזמינים בקטלוג דגמי Microsoft Foundry לצד דגמים מ-Meta, Mistral, Cohere ומיקרוסופט.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. עיצוב הפקודה  

"הקסם של מודלי שפה גדולים הוא שבאמצעות אימון למזעור שגיאת החיזוי הזאת על גבי כמויות עצומות של טקסט, המודלים לומדים בסופו של דבר מושגים שימושיים לחיזויים אלה. לדוגמה, הם לומדים מושגים כמו"(1):

* איך לאיית
* איך עובדת הדקדוק
* איך לפרפרז'
* איך לענות על שאלות
* איך לנהל שיחה
* איך לכתוב בשפות רבות
* איך לקודד
* וכו'.

#### איך לשלוט על מודל שפה גדול  
"מבין כל הקלטים למודל שפה גדול, המרשים ביותר הוא הפקודה בטקסט(1).

ניתן לגרום למודלי שפה גדולים להפיק פלט בכמה דרכים:

הוראה: אמור למודל מה אתה רוצה
השלמה: גרום למודל להשלים את תחילת מה שאתה רוצה
הדגמה: הראה למודל מה אתה רוצה, בין אם:
כמה דוגמאות בפקודה
מאות או אלפי דוגמאות בקבוצת אימון לכוונון עדין"



#### שלושה קווים מנחים בסיסיים ליצירת פקודות:

**הראה וספר**. היה ברור למה אתה רוצה באמצעות הוראות, דוגמאות, או שילוב ביניהן. אם אתה רוצה שהמודל יסדר רשימת פריטים בסדר אלפביתי או יסווג פסקה לפי רגש, הראה לו שזה מה שאתה רוצה.

**ספק נתונים איכותיים**. אם אתה מנסה לבנות מסווג או לגרום למודל לעקוב אחרי דפוס, וודא שיש מספיק דוגמאות. הקפד לקרוא את הדוגמאות שלך — המודל בדרך כלל חכם מספיק לזהות טעויות איות בסיסיות ולתת לך תגובה, אבל הוא גם עלול להניח שזה במתכוון וזה יכול להשפיע על התגובה.

**בדוק את ההגדרות שלך.** הגדרות הטמפרטורה ו-top_p שולטות כמה המודל דטרמיניסטי ביצירת תגובה. אם אתה מבקש תגובה שבה יש רק תשובה אחת נכונה, אז תרצה לשים את ההגדרות הנמוכות יותר. אם אתה מחפש תגובות מגוונות יותר, אז ייתכן שתרצה להעלות אותן. הטעות מספר אחת שאנשים עושים עם ההגדרות האלה היא להניח שהן שולטות ב"חכמת" או "יצירתיות" המודל.


מקור: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. הגש!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### חזור על אותו הקריאה, איך התוצאות משוות?  


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## לסכם טקסט  
#### אתגר  
סכמו טקסט על ידי הוספת 'tl;dr:' בסוף קטע הטקסט. שימו לב כיצד המודל מבין כיצד לבצע מספר משימות ללא הוראות נוספות. תוכלו להתנסות בהנחיות תיאוריות יותר מאשר tl;dr כדי לשנות את התנהגות המודל ולהתאים את הסיכום שתקבלו(3).  

עבודה אחרונה הראתה שיפורים משמעותיים בהרבה משימות ובנקודות בקרה ב-NLP על ידי אימון מקדים על מאגר גדול של טקסט ולאחר מכן כיוונון עדין למשימה מסוימת. בעוד שבדרך כלל הארכיטקטורה אינה תלויה במשימה, שיטה זו עדיין דורשת מאגרי נתונים מותאמים ספציפית למשימה המכילים אלפי או עשרות אלפי דוגמאות. לעומת זאת, בני אדם בדרך כלל יכולים לבצע משימה שפתית חדשה רק מכמה דוגמאות או מהוראות פשוטות - דבר שמערכות NLP עכשוויות עדיין מתקשות בו במידה רבה. כאן אנו מראים כי הגדלת המודל בצורה משמעותית משפרת הביצועים במשימות ללא תלות במשימה, לעיתים אפילו משיגה ביצועים תחרותיים לעומת שיטות כיוונון עדין מתקדמות קודמות.  



Tl;dr  


# תרגילים למספר מקרים שימושיים  
1. לסכם טקסט  
2. לסווג טקסט  
3. ליצור שמות חדשים למוצרים  


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## סיווג טקסט  
#### אתגר  
סווג פריטים לקטגוריות הניתנות בזמן החיזוי. בדוגמה הבאה, אנו מספקים גם את הקטגוריות וגם את הטקסט לסיווג בהנחיה (*playground_reference). 

פנייה מלקוח: שלום, אחת המפתחות במקלדת של המחשב הנייד שלי נשברה לאחרונה ואצטרך חלופה:

קטגוריה מסווגת:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## הפקת שמות מוצרים חדשים
#### אתגר
צור שמות מוצרים מתוך מילים לדוגמה. כאן אנו כוללים בהנחיה מידע על המוצר שעליו אנו הולכים לייצר שמות. אנו גם מספקים דוגמה דומה כדי להראות את התבנית שאנו רוצים לקבל. כמו כן, הגדרנו ערך טמפרטורה גבוה כדי להגדיל את האקראיות ואת התגובות החדשניות יותר.

תיאור מוצר: מכין שייקים ביתי
מילים בסיס: מהיר, בריא, קומפקטי.
שמות מוצרים: HomeShaker, Fit Shaker, QuickShake, Shake Maker

תיאור מוצר: זוג נעליים שיכולות להתאים לכל מידה של רגל.
מילים בסיס: מתאים, מתכוונן, אונמי-פיט.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# הפניות  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [שיטות עבודה מומלצות לכיול דק של מודלי GPT למיון טקסט](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# לקבלת עזרה נוספת  
[צוות המסחור של OpenAI](AzureOpenAITeam@microsoft.com) 


# תורמים
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
